# `json2vec` Hello World

This notebook trains a tiny model from an in-memory synthetic dataset.

In [1]:
import lightning.pytorch as lit
import torch
from rich.pretty import pprint

import json2vec as j2v

In [2]:
@j2v.shim(yields=True)
def hello_world_records(observation: dict, strata: j2v.Strata):

    records = [
        {"color": "red", "label": "warm"},
        {"color": "orange", "label": "warm"},
        {"color": "yellow", "label": "warm"},
        {"color": "blue", "label": "cool"},
        {"color": "green", "label": "cool"},
        {"color": "purple", "label": "cool"},
    ]

    yield from records

In [3]:
params = j2v.Hyperparameters(
    d_model=16,
    fields=j2v.Array(
        name="record",
        fields=[
            j2v.Category(name="color", query="[*].color", max_vocab_size=16),
            j2v.Category(name="label", query="[*].label", max_vocab_size=8, topk=[2]),
        ],
    ),
    target=j2v.Address("record", "label"),
    embed=j2v.Address("record"),
)


model = j2v.Architecture(
    hyperparameters=params,
    batch_size=4,
    optimizer=lambda module: torch.optim.AdamW(module.parameters(), lr=1e-2),
)

datamodule = j2v.StreamingDataModule.from_model(
    model,
    dataset=j2v.Dataset(processor=hello_world_records),
    num_workers=0,
    persistent_workers=False,
    pin_memory=False,
    file_buffer_size=1,
    observation_buffer_size=32,
    sample_rate=1.0,
)

2026-05-17 21:58:53.184 | INFO     | json2vec.architecture.root:__init__:128 - initialized JSON2Vec module


In [4]:
trainer = lit.Trainer(
    accelerator="mps",
    max_epochs=20,
    logger=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    # enable_checkpointing=False,
)

trainer.fit(model=model, datamodule=datamodule)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
2026-05-17 21:58:53.235 | INFO     | json2vec.data.datasets:dataloader:718 - building dataloader
2026-05-17 21:58:53.236 | INFO     | json2vec.data.datasets:__init__:650 - initialized batch dataset
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/grantham/Desktop/json2vec-oss/.venv/lib/python3.1

In [5]:
batch = [[{"color": "red"}], [{"color": "blue"}]]

In [6]:
pprint(model.predict(batch))

{
│   'record/label': {
│   │   'state': {
│   │   │   'valued': [0.9922986030578613, 0.9937534928321838],
│   │   │   'null': [0.001349523663520813, 0.0018522447207942605],
│   │   │   'padded': [0.002060377737507224, 0.00068658497184515],
│   │   │   'masked': [0.0030446010641753674, 0.0021226482931524515],
│   │   │   'other': [0.0012470345245674253, 0.0015852851793169975]
│   │   },
│   │   'content': {
│   │   │   'value': ['warm', 'cool'],
│   │   │   'probability': [0.9815559983253479, 0.9849434494972229],
│   │   │   'topk': [
│   │   │   │   [
│   │   │   │   │   {'label': 'warm', 'probability': 0.9815559983253479},
│   │   │   │   │   {'label': 'cool', 'probability': 0.018443796783685684}
│   │   │   │   ],
│   │   │   │   [
│   │   │   │   │   {'label': 'cool', 'probability': 0.9849434494972229},
│   │   │   │   │   {'label': 'warm', 'probability': 0.01505661103874445}
│   │   │   │   ]
│   │   │   ]
│   │   }
│   }
}

In [7]:
pprint(model.embed(batch))

{
│   'record': {
│   │   'embedding': [
│   │   │   [
│   │   │   │   -0.3342908024787903,
│   │   │   │   -0.11625323444604874,
│   │   │   │   0.36437729001045227,
│   │   │   │   0.11910861730575562,
│   │   │   │   -0.2952471673488617,
│   │   │   │   -0.3824993371963501,
│   │   │   │   0.05846869945526123,
│   │   │   │   -0.2511509358882904,
│   │   │   │   0.12299523502588272,
│   │   │   │   0.027829522266983986,
│   │   │   │   0.24338681995868683,
│   │   │   │   -0.3286813795566559,
│   │   │   │   0.434488981962204,
│   │   │   │   -0.00758998142555356,
│   │   │   │   0.23116610944271088,
│   │   │   │   -0.04856286197900772
│   │   │   ],
│   │   │   [
│   │   │   │   0.39176806807518005,
│   │   │   │   -0.2824479937553406,
│   │   │   │   -0.183658629655838,
│   │   │   │   0.03324776142835617,
│   │   │   │   0.033216312527656555,
│   │   │   │   -0.16914047300815582,
│   │   │   │   0.5399242043495178,
│   │   │   │   0.027449456974864006,
│   │   │   │   -0.34683525562286377,
│   │   │   │   -0.19156712293624878,
│   │   │   │   0.20664994418621063,
│   │   │   │   -0.18098612129688263,
│   │   │   │   0.2305300086736679,
│   │   │   │   0.07929231226444244,
│   │   │   │   -0.32860204577445984,
│   │   │   │   0.10028990358114243
│   │   │   ]
│   │   ]
│   }
}